In [ ]:
# Paso 1

In [ ]:
# Importar librerías necesarias
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Cargar el dataset apps.csv
apps = pd.read_csv("C:/Users/emata/Desktop/apps.csv")

In [ ]:
# Eliminar duplicados (si los hay)
apps = apps.drop_duplicates()


In [ ]:
# Imprimir el número total de apps (filas) tras limpiar duplicados
print("Total number of apps in the dataset = ", apps.shape[0])

In [ ]:
# Imprimir estadísticas descriptivas del dataset
print(apps.describe())

In [ ]:
# Vista previa del DataFrame
print(apps.head())

In [ ]:
# Paso 2

In [ ]:
# Lista de caracteres a eliminar
chars_to_remove = [',', '+', '$']

In [ ]:
# Lista de las columnas a limpiar
cols_to_clean = ['Installs', 'Price']

In [ ]:
# Loop para cada columna
for col in cols_to_clean:
    # Loop para cada caracter especial
    for char in chars_to_remove:
        # Reemplaza con una función lambda el carácter especial por un texto vacío ('')
        apps[col] = apps[col].apply(lambda x: x.replace(char, ''))
    
    # Convierte la columna a tipo flotante (float)
    apps[col] = apps[col].astype(float)

In [ ]:
# Revisar tipos de datos después de la limpieza
print(apps[['Installs', 'Price']].dtypes)

In [ ]:
# Revisar valores únicos para detectar residuos de caracteres especiales
print("Valores únicos en Installs (muestra):", apps['Installs'].unique()[:10])
print("Valores únicos en Price (muestra):", apps['Price'].unique()[:10])

In [ ]:
# Comprobar si todas las filas tienen valores numéricos válidos (sin NaNs tras la conversión)
print("Installs con NaN:", apps['Installs'].isna().sum())
print("Price con NaN:", apps['Price'].isna().sum())

In [ ]:
# Paso 3

In [ ]:
# Imprime el total de categorías únicas
num_categories = apps['Category'].nunique()
print('Number of categories =', num_categories)

In [ ]:
# Cuenta el número de aplicaciones en cada categoría y ordena de manera descendente
num_apps_in_category = apps['Category'].value_counts()

In [ ]:
# Muestra el resultado en una gráfica de barras
num_apps_in_category.plot(kind='bar', figsize=(12, 6))
plt.xlabel('Categoría')
plt.ylabel('Número de aplicaciones')
plt.title('Número de aplicaciones por categoría')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
# Paso 4

In [ ]:
# Calcular el promedio de calificación de las apps
avg_app_rating = apps['Rating'].mean()
print('Average app rating =', avg_app_rating)

In [ ]:
# Calcular el promedio de calificación por categoría
avg_rating_per_category = apps.groupby('Category')['Rating'].mean()

In [ ]:
# Visualizar en un histograma el comportamiento del Rating
apps['Rating'].hist(bins=20, figsize=(10, 6))
plt.xlabel('Rating')
plt.ylabel('Número de aplicaciones')
plt.title('Distribución de ratings de las apps')
plt.grid(False)
plt.show()

In [ ]:
# Paso 5

In [ ]:
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

sns.set_style("darkgrid")

In [ ]:
# Filtra filas donde los valores de 'Rating' y 'Size' no sean nulos
apps_with_size_and_rating_present = apps[apps['Rating'].notnull() & apps['Size'].notnull()]

In [ ]:
# Filtra las categorías con al menos 250 apps
large_categories = apps_with_size_and_rating_present.groupby('Category').filter(lambda x: len(x) >= 250)

In [ ]:
# Gráfica size vs. rating
plt1 = sns.jointplot(x=large_categories['Size'], y=large_categories['Rating'])

In [ ]:
# Selecciona las apps de paga 'Type' = 'Paid'
paid_apps = apps[apps['Type'] == 'Paid']

In [ ]:
# Gráfica price vs. rating de las aplicaciones de paga
plt2 = sns.jointplot(x=paid_apps['Price'], y=paid_apps['Rating'])
plt.show()

In [ ]:
# Paso 6

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots()
fig.set_size_inches(15, 8)

# Lista de categorías populares
popular_app_cats = apps[apps.Category.isin([
    'GAME', 'FAMILY', 'PHOTOGRAPHY',
    'MEDICAL', 'TOOLS', 'FINANCE',
    'LIFESTYLE', 'BUSINESS'
])]

# Examina la tendencia de precio graficando el Precio por Categoría
ax = sns.stripplot(x=popular_app_cats['Category'], y=popular_app_cats['Price'], jitter=True, linewidth=1)
ax.set_title('App pricing trend across categories')

# Selecciona las apps con un precio mayor a 200
apps_above_200 = apps[apps['Price'] > 200]
apps_above_200[['Category', 'App', 'Price']]


In [ ]:
# Paso 7

In [ ]:
# Agrupa por tipo de app (Free o Paid) y calcula estadísticas relevantes
apps_by_type = apps.groupby('Type').agg({
    'Rating': 'mean',
    'Size': 'mean',
    'Reviews': 'mean',
    'Installs': 'mean',
    'Price': 'mean'
}).reset_index()

print(apps_by_type)


In [ ]:
# Paso 8

In [ ]:
# Carga el archivo user_reviews.csv
reviews_df = pd.read_csv("C:/Users/emata/Desktop/user_reviews.csv")

In [ ]:
# Une los dos DataFrames (join)
merged_df = pd.merge(apps, reviews_df, on='App')

In [ ]:
# Elimina los valores nulos (NA) de las columnas Sentiment y Review
merged_df = merged_df.dropna(subset=['Sentiment', 'Review'])


In [ ]:
# Grafica la polaridad de sentimientos para apps gratuitas y de paga
sns.set_style('ticks')
fig, ax = plt.subplots()
fig.set_size_inches(11, 8)

ax = sns.boxplot(x='Type', y='Sentiment_Polarity', data=merged_df)
ax.set_title('Sentiment Polarity Distribution')